# 教学版 BPE 分词器 (Byte-Pair Encoding)

> 父文档：[← 分词器总览](README.ipynb)

本笔记本旨在通过交互式代码讲解 BPE (Byte-Pair Encoding) 的核心算法。读完本篇后，你将能无缝理解源码中的 [core/tokenizer/bpe.py](../../core/tokenizer/bpe.py) 实现。

## 1. 核心思想

BPE 的核心是**频次驱动的迭代合并**：
1. **初始化**：将单词切分成字符，并添加词尾标记 `</w>`。
2. **统计**：寻找语料中相邻出现频次最高的 Pair（两个 Token）。
3. **合并**：将该 Pair 合并成一个新的 Token。
4. **循环**：重复上述过程，直到词表达到目标大小。

---

## 2. 算法步骤拆解

我们先定义一些基础工具，它们对应源码中的核心逻辑。

### 2.1 预分词与初始化
在做 BPE 之前，我们需要先将文本粗略地切分为“单词”或“符号”块。

源码对应：[`_pretokenize`](../../core/tokenizer/bpe.py#L22) 和 [`_word_to_symbols`](../../core/tokenizer/bpe.py#L30)


In [2]:
import re
from collections import Counter

# 与 GPT-2 一致的预切分正则
_PRETOKEN_RE = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+""", re.UNICODE)
END_OF_WORD = "</w>"

def pretokenize(text):
    return [m.group(0) for m in _PRETOKEN_RE.finditer(text) if m.group(0)]

def word_to_symbols(word):
    # 将单词拆解为字符序列，并加上词尾标记
    return tuple(list(word) + [END_OF_WORD])

test_text = "low lower newest"
pieces = pretokenize(test_text)
print(f"预切分结果: {pieces}")

# 初始化统计词频
vocab = Counter()
for p in pieces:
    vocab[word_to_symbols(p)] += 1

print("\n初始化后的 Vocab (symbols):")
for k, v in vocab.items():
    print(f"  {k}: {v}")

预切分结果: ['low', ' lower', ' newest']

初始化后的 Vocab (symbols):
  ('l', 'o', 'w', '</w>'): 1
  (' ', 'l', 'o', 'w', 'e', 'r', '</w>'): 1
  (' ', 'n', 'e', 'w', 'e', 's', 't', '</w>'): 1


### 2.2 统计相邻 Pair 频次
算法的核心是找到最应该合并的两个 Token。

源码对应：[`_get_pair_counts`](../../core/tokenizer/bpe.py#L34)


In [3]:
def get_pair_counts(vocab):
    counts = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            counts[pair] += freq
    return counts

pair_counts = get_pair_counts(vocab)
print("相邻 Pair 统计 (前 5 个):")
print(pair_counts.most_common(5))

相邻 Pair 统计 (前 5 个):
[(('l', 'o'), 2), (('o', 'w'), 2), (('w', 'e'), 2), (('w', '</w>'), 1), ((' ', 'l'), 1)]


### 2.3 执行合并操作
一旦选定 `(a, b)`，我们需要遍历所有单词，将其中连续出现的 `a, b` 替换为 `ab`。

源码对应：`_merge_in_word`

In [4]:
def merge_in_word(symbols, pair):
    a, b = pair
    new_token = a + b
    out = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
            out.append(new_token)
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)

# 假设我们要合并 ('e', 's')
best_pair = ('e', 's')
print(f"尝试合并: {best_pair}")
sample_word = word_to_symbols("newest")
print(f"合并前: {sample_word}")
print(f"合并后: {merge_in_word(sample_word, best_pair)}")

尝试合并: ('e', 's')
合并前: ('n', 'e', 'w', 'e', 's', 't', '</w>')
合并后: ('n', 'e', 'w', 'es', 't', '</w>')


### 2.3 执行合并操作
一旦选定 `(a, b)`，我们需要遍历所有单词，将其中连续出现的 `a, b` 替换为 `ab`。

源码对应：[`_merge_in_word`](../../core/tokenizer/bpe.py#L42)


In [5]:
# 初始化
current_vocab = dict(vocab)
merges = []
token_to_id = {"<|endoftext|>": 0, "<|unk|>": 1} # 特殊 token

# 初始字符进词表
for symbols in current_vocab:
    for char in symbols:
        if char not in token_to_id:
            token_to_id[char] = len(token_to_id)

# 迭代合并 (假设合并 5 次)
for i in range(5):
    pairs = get_pair_counts(current_vocab)
    if not pairs: break
    best_pair, freq = pairs.most_common(1)[0]
    
    merged_token = best_pair[0] + best_pair[1]
    if merged_token not in token_to_id:
        token_to_id[merged_token] = len(token_to_id)
    
    merges.append(best_pair)
    current_vocab = {merge_in_word(s, best_pair): f for s, f in current_vocab.items()}
    print(f"Step {i+1}: 合并 {best_pair}, 词表大小 {len(token_to_id)}")

print("\n训练出的 Merges 规则:")
print(merges)

Step 1: 合并 ('l', 'o'), 词表大小 13
Step 2: 合并 ('lo', 'w'), 词表大小 14
Step 3: 合并 ('low', '</w>'), 词表大小 15
Step 4: 合并 (' ', 'low'), 词表大小 16
Step 5: 合并 (' low', 'e'), 词表大小 17

训练出的 Merges 规则:
[('l', 'o'), ('lo', 'w'), ('low', '</w>'), (' ', 'low'), (' low', 'e')]


### 4.1 编码逻辑
编码时，对输入文本做相同的预分词，然后**严格按照训练时的 merges 顺序**进行合并。

源码对应：[`BPETokenizer._bpe`](../../core/tokenizer/bpe.py#L137)


In [6]:
def bpe_encode(word, merge_rank):
    symbols = list(word_to_symbols(word))
    while True:
        # 在当前 symbols 中寻找 rank 最小（即训练中最早出现）的可用 merge
        best_pair = None
        best_rank = float('inf')
        best_idx = -1
        
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            rank = merge_rank.get(pair)
            if rank is not None and rank < best_rank:
                best_rank = rank
                best_pair = pair
                best_idx = i
        
        if best_pair is None:
            break
            
        # 执行合并
        a, b = best_pair
        symbols = symbols[:best_idx] + [a + b] + symbols[best_idx + 2:]
        
    return symbols

# 准备编码权重
merge_rank = {p: i for i, p in enumerate(merges)}

test_word = "lower"
encoded_tokens = bpe_encode(test_word, merge_rank)
print(f"词汇: {test_word}")
print(f"BPE 切分产物: {encoded_tokens}")

词汇: lower
BPE 切分产物: ['low', 'e', 'r', '</w>']


### 4.2 解码逻辑
解码非常简单：直接将所有 Token 拼接，然后全局替换 `</w>` 为空即可。

源码对应：[`BPETokenizer.decode`](../../core/tokenizer/bpe.py#L161)


In [7]:
def decode(tokens):
    text = "".join(tokens)
    return text.replace(END_OF_WORD, "")

print(f"解码结果: {decode(encoded_tokens)}")

解码结果: lower


## 5. 工程实现提示

在源码 [`core/tokenizer/bpe.py`](../../core/tokenizer/bpe.py) 中，你会看到以下额外的工程细节：
- **`BaseTokenizer` 继承**：统一了 `save`/`load` 接口。
- **ID 映射**：使用 `token_to_id` 和 `id_to_token` 处理整数索引。
- **`<|unk|>` 处理**：对于训练语料中从未见过的字符，映射到 `unk_id`。

---

## 6. 与 GPT-2 的关系

本方案是**字符级 (Character-level) BPE**，主要用于算法演示。
真正的 GPT-2 使用的是 **字节级 (Byte-level) BPE**，它将文本转为 UTF-8 字节流后再处理，可以解决 Unicode 扩展带来的词表爆炸问题。

想深入了解字节级变体？请阅读：[02_byte_level_bpe.md](02_byte_level_bpe.md)

## 7. 延伸阅读与参考资料

### 核心论文 (Paper)
- **BPE 开山之作**: Sennrich et al., 2016. *Neural Machine Translation of Rare Words with Subword Units.* [arXiv:1508.07909](https://arxiv.org/abs/1508.07909)
- **GPT-2 技术报告**: Radford et al., 2019. *Language Models are Unsupervised Multitask Learners* (介绍字节级 BPE). [Link](https://openai.com/blog/better-language-models/)

### 优质博客 (Blog)
- **Hugging Face Course**: *Byte-Pair Encoding tokenization*. [NLP Course Chapter 6](https://huggingface.co/learn/nlp-course/chapter6/5)
- **Andrej Karpathy**: *Let's build the GPT Tokenizer*. [YouTube Tutorial](https://www.youtube.com/watch?v=zduSFxRajkE) (推荐，非常直观)

### 代码库参考 (Code)
- **OpenAI GPT-2 官方实现**: [openai/gpt-2/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- **Hugging Face Tokenizers**: 高性能 Rust 实现 [huggingface/tokenizers](https://github.com/huggingface/tokenizers)

---
> 父文档：[← 分词器总览](README.md)